### **WorldMove Trajectories – Traffic Data Simulation**

### **i. Background**

This dataset contains high-resolution mobility trajectories collected from WorldMove, representing the movement of individual agents across space and time. Each agent’s location (latitude and longitude) is recorded at sequential timestamps, allowing us to reconstruct travel patterns within the study area.

We use these trajectories as **traffic proxies**: by analyzing the density, frequency, and flow of agents through different wards and hours, we can approximate **temporal transit demand** and **service pressure points**. ~~This data will later be integrated with ward-level service data to build **spatio-temporal sequences**, which are suitable for predictive modeling approaches such as LSTM networks.~~

The `.npz` format stores the trajectories as compressed arrays, with keys typically representing:

* `lon` / `lat` → coordinates of each movement point
* `time` → timestamps
* `agent_id` → unique identifier per moving individual

By combining this trajectory data with existing transit service metrics, we aim to **simulate realistic traffic patterns** and estimate **ETAs and traffic congestion**.

### **ii. Libraries**

In [2]:
import pandas as pd
import numpy as np
import zipfile
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial.distance import cdist
import osmnx as ox
import polars as pl
from tqdm import tqdm
from functools import lru_cache
import warnings
import os
import gtfs_kit as gk
warnings.filterwarnings("ignore", category=UserWarning)
print("Imports complete.")

Imports complete.


### **iii. Data**

#### **a. WorldMove Trajectories:**
  * A trajectory is the **sequence of spatial positions** for a given agent over time.
  * These sequences can be used to study **travel patterns, mobility trends, peak activity periods, or simulate flows** in transportation models.
* We load mobility trajectory data collected from WorldMove, which tracks individual movements across space and time. Each record represents an agent’s location (latitude/longitude) at a specific timestamp.
* **`.npz` file architecture:**

  * An `.npz` file is a **compressed NumPy archive** containing multiple arrays.
  * Each array is stored as a **key-value pair** where the key is a string and the value is a NumPy array.
  * Typical keys in trajectory datasets:

    * `lon` / `lat`: coordinates of the agent.
    * `time`: timestamps for each location record.
    * `agent_id` / `user_id`: unique identifier for each moving individual.
    * Other optional metadata arrays depending on the dataset.

In [3]:
# Load the .npz file
traj = np.load('/home/dataopske/Desktop/jav/data/raw/worldmove/worldmove_trajectories.npz', allow_pickle=True)

# List keys inside the archive
print("Keys in the file:", traj.files)

# Check shapes and types
for key in traj.files:
    print(f"{key}: shape={traj[key].shape}, dtype={traj[key].dtype}")
    # Load data

grid_dict = traj['grid'].item()
poi_data = traj['poi']      # (30, 48, 34) - 30 cells, 48 timesteps, 34 POI types
pop_data = traj['pop']       # (30, 48) - 30 cells, 48 timesteps
traj_data = traj['traj']     # (104538, 48) - 104,538 agents, 48 timesteps

print(f"Total grid cells in dict: {len(grid_dict)}")  # Should be 1440
print(f"Agents tracked: {traj_data.shape[0]}")
print(f"Timesteps: {traj_data.shape[1]}")

Keys in the file: ['grid', 'poi', 'pop', 'traj']
grid: shape=(), dtype=object
poi: shape=(30, 48, 34), dtype=float32
pop: shape=(30, 48), dtype=float64
traj: shape=(104538, 48), dtype=int64
Total grid cells in dict: 1440
Agents tracked: 104538
Timesteps: 48


* **Next steps:** Once loaded, these arrays can be **converted into a structured DataFrame** or GeoDataFrame for merging with ward-level service data, creating spatio-temporal sequences for LSTM modeling, or calculating aggregate mobility metrics.

#### **b. Wards data**
We open the Kenya Wards ZIP archive and list its contents to identify the shapefiles. These boundaries will be used for spatial joins with service and trajectory data, enabling downstream GIS analysis.


In [4]:
# Open the ZIP file in read mode ('r')
zip_path = '/home/dataopske/Desktop/jav/data/raw/kenyawards/kenya_wards.zip'  # Replace with your ZIP file path
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_list = zip_ref.namelist()  # Returns a list of file paths (including subdirs)
    print(file_list)  # e.g., ['data.csv', 'folder/image.jpg']
    pass

kenya_wards = gpd.read_file(f"zip://{zip_path}!Kenya_Wards/kenya_wards.shp")

['Kenya_Wards/', 'Kenya_Wards/kenya_wards.cpg', 'Kenya_Wards/kenya_wards.dbf', 'Kenya_Wards/kenya_wards.prj', 'Kenya_Wards/kenya_wards.qpj', 'Kenya_Wards/kenya_wards.shp', 'Kenya_Wards/kenya_wards.shx']


The zip contains multiple files, we're interested in the `Kenya_Wards/kenya_wards.shp`

In [5]:
# Load Nairobi boundary
nairobi_gdf = gpd.read_file('/home/dataopske/Desktop/jav/data/raw/supportdata/nairobi.json')

# Clip wards to Nairobi
wards_nairobi_gdf = gpd.clip(kenya_wards, nairobi_gdf)


#### **c. Extracting and Preparing Grid Cell Centroids (WorldMoveData)**

1. **Extract cell centroids from `grid_dict`**  
   We loop through each entry in the grid dictionary, separating the cell IDs and their corresponding coordinates (longitude, latitude).  
   This allows us to work with the grid data in a structured format suitable for geospatial analysis.

2. **Convert coordinates to a NumPy array**  
   Storing the coordinates as a NumPy array makes it easier to perform vectorized operations and integrate with GeoPandas.

3. **Create a GeoDataFrame of grid cells**  
   Using `GeoDataFrame`, we transform the cell centroids into geometric points.  
   This gives each cell a spatial representation and sets the coordinate reference system (CRS) to WGS84 (`EPSG:4326`), the standard for lat/lon data.

4. **Transform to match ward CRS (`EPSG:32737`)**  
   To overlay or spatially join these grid cells with the Kenya wards shapefile, we reproject the grid GeoDataFrame to the same CRS as the wards.  
   This ensures spatial compatibility for any future geospatial operations.


In [6]:
# Extract cell centroids from grid dictionary
cell_ids = []
cell_coords = []

for cell_id_str, coords in grid_dict.items():
    cell_ids.append(int(cell_id_str))
    cell_coords.append(coords)  # [lon, lat]

cell_coords = np.array(cell_coords)

# Create GeoDataFrame of grid cells
grid_gdf = gpd.GeoDataFrame(
    {'cell_id': cell_ids},
    geometry=[Point(lon, lat) for lon, lat in cell_coords],
    crs='EPSG:4326'
)

# Transform to match ward CRS (EPSG:32737)
grid_gdf = grid_gdf.to_crs('EPSG:32737')

print(f"Grid cells created: {len(grid_gdf)}")
grid_gdf.head()

Grid cells created: 1440


,cell_id,geometry
0,0,POINT (240575.158 9871043.776)
1,1,POINT (241316.295 9870796.395)
2,2,POINT (242856.284 9871107.748)
3,3,POINT (243795.629 9870820.386)
4,4,POINT (245160.049 9871506.409)


#### **c. Mapping Grid Cells to Wards**

1. **Align CRS with wards shapefile**  
   We reproject the grid GeoDataFrame to the same CRS as the full wards dataset (`ward_full_gdf.crs`) to ensure accurate spatial operations.

2. **Spatial join: cells → wards**  
   Using `geopandas.sjoin`, we assign each grid cell to the ward it falls within.  
   - `how='left'` keeps all grid cells initially.  
   - `predicate='within'` ensures we only assign wards to cells that lie inside a ward polygon.

3. **Remove cells outside Nairobi**  
   Cells that do not fall within any ward will have `NaN` for the ward column.  
   We drop these to focus only on grid cells that are inside Nairobi.

4. **Verify mapping**  
   Print how many cells successfully mapped to wards, giving a quick sanity check on coverage.


In [7]:
# Now you can use wards_nairobi_gdf in place of ward_full_gdf
grid_gdf = grid_gdf.to_crs(wards_nairobi_gdf.crs)

cell_to_ward = gpd.sjoin(
    grid_gdf,
    wards_nairobi_gdf[['ward', 'geometry']],
    how='left',
    predicate='within'
)

# Drop cells outside Nairobi
cell_to_ward = cell_to_ward.dropna(subset=['ward'])
print(f"Cells mapped to wards: {cell_to_ward['ward'].notna().sum()} / {len(cell_to_ward)}")

Cells mapped to wards: 738 / 738


A total of 738/738 cells were mapped to nairobi bounds

5. **Visualisation of the cells**

In [8]:
import plotly.graph_objects as go

# Check what CRS cell_to_ward is currently in
print(f"Current CRS: {cell_to_ward.crs}")

# Calculate centroids in projected CRS (EPSG:32737 for Nairobi)
if cell_to_ward.crs.to_epsg() == 4326:
    # If already in WGS84, convert to projected first
    cell_projected = cell_to_ward.to_crs(epsg=32737)
else:
    cell_projected = cell_to_ward.copy()

# Calculate centroids in projected CRS
cell_projected['centroid'] = cell_projected.geometry.centroid

# Convert centroids to WGS84 for plotting
cell_centroids_wgs = cell_projected.set_geometry('centroid').to_crs(epsg=4326)
cell_centroids_wgs['lon'] = cell_centroids_wgs.geometry.x
cell_centroids_wgs['lat'] = cell_centroids_wgs.geometry.y

fig = go.Figure()

# Add grid cell points
fig.add_trace(go.Scattermap(
    lon=cell_centroids_wgs['lon'],
    lat=cell_centroids_wgs['lat'],
    mode='markers',
    marker=dict(
        size=8,
        color=cell_centroids_wgs['ward'].astype('category').cat.codes,
        colorscale='Viridis',
        opacity=0.7
    ),
    text=cell_centroids_wgs['ward'],
    hovertemplate='<b>%{text}</b><extra></extra>',
    name='Grid Cells'
))

# Add ward boundaries
wards_wgs = wards_nairobi_gdf.to_crs(epsg=4326)

for idx, row in wards_wgs.iterrows():
    if row.geometry.geom_type == 'Polygon':
        coords = list(row.geometry.exterior.coords)
    else:
        coords = list(row.geometry.geoms[0].exterior.coords)
    
    lons, lats = zip(*coords)
    
    fig.add_trace(go.Scattermap(
        lon=lons,
        lat=lats,
        mode='lines',
        line=dict(color='black', width=1.5),
        showlegend=False,
        hoverinfo='skip'
    ))

fig.update_layout(
    map=dict(
        style='carto-positron',
        center=dict(lat=-1.286389, lon=36.817223),
        zoom=10
    ),
    title="Grid Cells Mapped to Nairobi Wards",
    height=700,
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()

Current CRS: EPSG:4326


**Interpretation of Mapped Grid Cells**

Each point on the map represents the **centroid of a grid cell** that lies within Nairobi. By spatially joining these centroids with ward boundaries, we now know which ward each grid cell belongs to.  

These mapped cells serve several purposes:  
1. **Spatial indexing** – they break Nairobi into uniform units for analysis, allowing metrics (population, cases, services) to be aggregated or analyzed at a finer resolution than wards.  
2. **Visualization** – we can clearly see the distribution of the grid across the city and how it overlaps with administrative wards.  
3. **Data integration** – any georeferenced data (e.g., health facilities, survey points, environmental measurements) can be linked to these grid cells for ward-level or sub-ward analysis.  
4. **Analytical flexibility** – they provide a framework for spatial statistics, such as density mapping, hotspot detection, or interpolation of measurements across the city.  

**Grid Cells as "Anchor Points"**

Each grid cell centroid acts as an **anchor point** across Nairobi. Agents or events can be tracked moving from cell to cell, making it easy to:  
- Monitor movement between locations.  
- Aggregate data or events within cells.  
- Link activities to wards for analysis.  
- Model spatial processes like traffic, disease spread, or service accessibility.  

In essence, these cells provide a **structured, granular framework** for city-wide spatial tracking and analysis.

#### **d. Converting Timesteps to Real Hours**

From the WorldMove documentation; **48 timesteps = 30-minute intervals** (24 hours × 2).  
Timestep 0 → 00:00, Timestep 1 → 00:30, and so on.  
We then map key hours (06:00, 09:00, 15:00) to their corresponding timesteps  
to focus our analysis on morning, mid-morning, and afternoon patterns.


In [9]:
# Check if 48 timesteps = 48 half-hours (30-min intervals)
# Pop at t=0 vs t=1 shows significant change, suggesting different times of day

# Assumption: 48 timesteps = 30-minute intervals (24 hours × 2)
# Timestep 0 = 00:00, Timestep 1 = 00:30, ..., Timestep 12 = 06:00, etc.

timestep_to_hour = {
    t: (t // 2) for t in range(48)
}

# Map our target hours to timesteps
target_hours = [6, 9, 15]
target_timesteps = {
    6: [12, 13],   # 06:00 and 06:30
    9: [18, 19],   # 09:00 and 09:30
    15: [30, 31]   # 15:00 and 15:30
}

print("Hour to Timestep mapping:")
for hour, timesteps in target_timesteps.items():
    print(f"  Hour {hour}: timesteps {timesteps}")

Hour to Timestep mapping:
  Hour 6: timesteps [12, 13]
  Hour 9: timesteps [18, 19]
  Hour 15: timesteps [30, 31]


#### **e. Extracting Agent Trips from Trajectory Data**

**What We're Doing**

Each row in `traj_data` represents an **agent**, and each column is the **grid cell** they occupy at a specific timestep.  
We process these trajectories to identify **movements between cells**, filtering only those that occur **within Nairobi wards**.

This cell processes raw agent movement data (trajectories) and converts it into a structured dataset of trips between wards. Each agent's trajectory is a sequence of grid cells they occupy at each timestep throughout the day. We extract meaningful trips by:

1. **Identifying movement**: Detecting when an agent moves from one cell to another (skipping stationary periods)
2. **Mapping to wards**: Converting cell IDs to their corresponding administrative wards using a spatial lookup
3. **Filtering valid trips**: Only keeping trips where both origin and destination cells are within mapped Nairobi wards
4. **Adding temporal context**: Recording the hour of day for each trip using the timestep-to-hour mapping

**Why Use Polars?**
We use **Polars** (`pl.DataFrame`) instead of pure Pandas for the final DataFrame creation because:

- **Faster DataFrame construction**: Polars is optimized for creating DataFrames from large lists of dictionaries (3-5x faster than Pandas)
- **Memory efficiency**: Better memory management when handling 1M+ rows
- **Seamless conversion**: `.to_pandas()` allows us to convert back to Pandas for compatibility with existing analysis code
- **Modern performance**: Built in Rust, designed for speed with large datasets

For the core processing loop, we use **dictionary lookups** (`cell_to_ward_dict.get()`) instead of DataFrame queries because:
- Dictionary lookup is O(1) - constant time regardless of dataset size
- Pandas `.loc` queries are O(n) - slow when repeated millions of times
- This optimization provides a **5-10x speedup** over the original `.loc` approach



In [10]:
import polars as pl
import numpy as np

print("Processing trajectories...")

# Create cell lookup
cell_to_ward_dict = cell_to_ward.set_index('cell_id')['ward'].to_dict()
valid_cells = set(cell_to_ward_dict.keys())

trips_list = []

for agent_id in range(traj_data.shape[0]):
    if agent_id % 10000 == 0:
        print(f"  Processing agent {agent_id}/{traj_data.shape[0]}")
    
    agent_trajectory = traj_data[agent_id, :]
    
    # Vectorized operations
    origins = agent_trajectory[:-1]
    dests = agent_trajectory[1:]
    moved = origins != dests
    
    for t in np.where(moved)[0]:
        origin_cell = origins[t]
        dest_cell = dests[t]
        
        if origin_cell not in valid_cells or dest_cell not in valid_cells:
            continue
        
        origin_ward = cell_to_ward_dict.get(origin_cell)
        dest_ward = cell_to_ward_dict.get(dest_cell)
        
        if origin_ward is None or dest_ward is None:
            continue
        
        trips_list.append({
            'agent_id': agent_id,
            'timestep': t,
            'hour': timestep_to_hour[t],
            'origin_ward': origin_ward,
            'dest_ward': dest_ward,
            'origin_cell': origin_cell,
            'dest_cell': dest_cell
        })

# Use Polars for faster DataFrame creation
trips_df = pl.DataFrame(trips_list).to_pandas()

print(f"\nTotal trips extracted: {len(trips_df):,}")
print(f"Unique agents: {trips_df['agent_id'].nunique():,}")
print(f"Hours covered: {sorted(trips_df['hour'].unique())}")

Processing trajectories...
  Processing agent 0/104538


  Processing agent 10000/104538
  Processing agent 20000/104538
  Processing agent 30000/104538
  Processing agent 40000/104538
  Processing agent 50000/104538
  Processing agent 60000/104538
  Processing agent 70000/104538
  Processing agent 80000/104538
  Processing agent 90000/104538
  Processing agent 100000/104538

Total trips extracted: 1,150,948
Unique agents: 88,483
Hours covered: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23)]



**Understanding the Output**
```
Total trips extracted: 1,150,948
Unique agents: 88,483
Hours covered: [0, 1, 2, ..., 23]
```

**What this tells us:**
- **1.15M trips**: Total number of meaningful movements between wards across all agents and all hours
- **88,483 unique agents** (~85% of 104,538 total): Only agents who made at least one trip within mapped Nairobi wards are counted. The remaining ~16k agents either:
  - Stayed stationary all day
  - Only traveled outside mapped ward boundaries
  - Moved between unmapped cells
- **24 hours covered**: Trips occur throughout the entire day (midnight to 11 PM), allowing for temporal equity analysis

**Key filtering decisions:**
- Excluded **702 unmapped cells**: Grid cells in the trajectory data that don't correspond to any Nairobi ward (likely outside city boundaries or in unmapped areas)
- Excluded **~288k trips**: Movements involving these unmapped cells (about 20% of raw movements)
- This ensures our analysis focuses on **transit equity within Nairobi's administrative boundaries**

**Data quality check:**
- All trips have valid origin and destination wards (0 None values)
- 88 unique wards represented in both origins and destinations
- This confirms complete spatial coverage of Nairobi's ward structure

In [11]:
trips_df.head(10)

,agent_id,timestep,hour,origin_ward,dest_ward,origin_cell,dest_cell
0,0,25,12,Roysambu Ward,Zimmerman Ward,265,266
1,0,28,14,Zimmerman Ward,Roysambu Ward,266,265
2,0,32,16,Roysambu Ward,Zimmerman Ward,265,217
3,0,36,18,Zimmerman Ward,Kiambu Township Ward,217,169
4,1,20,10,Utalii Ward,Babandogo,455,454
5,1,22,11,Babandogo,Roysambu Ward,454,453
6,1,23,11,Roysambu Ward,Roysambu Ward,453,405
7,1,31,15,Roysambu Ward,Roysambu Ward,405,404
8,1,32,16,Roysambu Ward,Roysambu Ward,404,452
9,1,37,18,Roysambu Ward,Karura Ward,452,451


#### **f. Loading the Nairobi Road Network**

Here we **load or generate the Nairobi road network**, which we’ll later use to link agent movements to actual street geometry.
This step ensures we have a **realistic street graph** for analyzing mobility patterns, calculating travel distances, or visualizing flows.

1. **We first check** if a cached version of the road network (`nairobi_drive.graphml`) already exists locally.

   * If it does, we simply **load it** to save time and avoid re-downloading.
2. **If not found**, we **download the road network** for Nairobi directly from **OpenStreetMap (OSM)** using OSMnx.

   * We define a **bounding box** covering Nairobi (`west=36.6, south=-1.5, east=37.0, north=-1.1`) and request the **drivable road network** (`network_type='drive'`).
3. **We then save** the network to disk so future runs load instantly.

In [12]:
graph_path = "/home/dataopske/Desktop/jav/data/processed/nairobi_drive.graphml"
if os.path.exists(graph_path):
    print("Loading cached Nairobi road network...")
    G_drive = ox.load_graphml(graph_path)
else:
    print("Downloading and saving Nairobi road network (first-time only)...")
    west, south, east, north = 36.6, -1.5, 37.0, -1.1
    G_drive = ox.graph_from_bbox(bbox=(west, south, east, north), network_type='drive')
    ox.save_graphml(G_drive, graph_path)
    print(f"Graph saved: {len(G_drive.nodes)} nodes, {len(G_drive.edges)} edges")
print("Graph loaded successfully.")

Loading cached Nairobi road network...
Graph loaded successfully.


> This graph forms the **base infrastructure layer** for all later spatial operations, such as matching trajectories to roads, measuring accessibility, or modeling network flow.

#### **g. Prepare Cell Centroids (WGS84 and UTM)**

Here we **prepare coordinate lookup tables** for each grid cell, both in **geographic (WGS84)** and **projected (UTM)** coordinate systems.
These lookups let us **quickly access the spatial position** of any grid cell by its ID during movement reconstruction or distance computation.

1. **We start by converting** the grid GeoDataFrame to WGS84 (`EPSG:4326`), the standard latitude–longitude format used for global mapping and visualization.
2. **We extract** each cell’s `(lon, lat)` pair and store it in a NumPy array for fast access.
3. **We create two lookup dictionaries**:

   * `cell_coord_lookup_wgs`: for geographic coordinates (used in maps and visualizations).
   * `cell_coord_lookup_utm`: for UTM coordinates (used in distance and network calculations where metric units are needed).
4. **We print summary stats** to confirm that all grid cells are included and ready for downstream geospatial operations.


In [13]:
print("Prepping cell coordinates...")
grid_gdf_wgs = grid_gdf.to_crs(epsg=4326)
cell_coords_wgs = np.array([[pt.x, pt.y] for pt in grid_gdf_wgs.geometry])
cell_coord_lookup_wgs = dict(zip(grid_gdf['cell_id'], cell_coords_wgs))

cell_coord_lookup_utm = dict(zip(
    grid_gdf['cell_id'],
    [(pt.x, pt.y) for pt in grid_gdf.geometry]
))
print("Coordinate lookups created.")
print(f"WGS84 lookup keys: {len(cell_coord_lookup_wgs)}")
print(f"UTM lookup keys: {len(cell_coord_lookup_utm)}")

Prepping cell coordinates...
Coordinate lookups created.
WGS84 lookup keys: 1440
UTM lookup keys: 1440


> This setup ensures efficient coordinate retrieval when we later compute **distances, route paths**, or **visualize trajectories** on the Nairobi map.

#### **h. Snapping Grid Cells to Road Network Nodes**

Here we **link each grid cell to its nearest road node** within the Nairobi street network.
This step connects our **abstract grid-based trajectories** to the **real road infrastructure**, enabling realistic movement modeling and distance calculations.

1. **We start by identifying** all unique grid cells that appear as either origins or destinations in the trips dataset.

   * This ensures we only process cells that are actually used in the mobility data.
2. **For each unique cell**, we:

   * Retrieve its geographic coordinates `(lon, lat)` from our lookup table.
   * Use **OSMnx’s `nearest_nodes()`** to find the closest drivable road node within the network graph.
   * Store this mapping in a dictionary (`cell_to_node[cell] = node`) for quick lookups later.
3. **We handle missing cases gracefully** — if a cell lies outside the road network or lookup fails, we assign it `None` instead of breaking execution.

In [14]:
print("Snapping grid cells to road nodes...")
unique_cells = np.unique(np.concatenate([
    trips_df['origin_cell'].unique(),
    trips_df['dest_cell'].unique()
]))

cell_to_node = {}
for cell in tqdm(unique_cells, desc="Snapping cells"):
    lonlat = cell_coord_lookup_wgs.get(cell)
    if lonlat is not None:
        lon, lat = lonlat
        try:
            node = ox.nearest_nodes(G_drive, lon, lat)
            cell_to_node[cell] = node
        except Exception:
            cell_to_node[cell] = None
    else:
        cell_to_node[cell] = None

snapped_count = sum(n is not None for n in cell_to_node.values())
print(f"Snapped {snapped_count:,}/{len(unique_cells):,} cells to nodes ({100*snapped_count/len(unique_cells):.1f}% success)")
print("Snapping complete.")

Snapping grid cells to road nodes...


Snapping cells:   0%|          | 0/738 [00:00<?, ?it/s]

Snapping cells: 100%|██████████| 738/738 [04:14<00:00,  2.90it/s]

Snapped 738/738 cells to nodes (100.0% success)
Snapping complete.


> This process effectively **anchors every active grid cell to the physical road system**, bridging the gap between **simulated trajectories** and **real-world geography** — a crucial step before computing travel paths or flow distances.

#### **i. Defining Distance Calculator Functions**

Here we **define helper functions** to calculate the **travel distance between origin and destination cells**, using either the **road network** or **straight-line (great-circle)** fallback.

1. **We begin by creating a cached network distance function**

   * The function `route_distance_cached()` uses **OSMnx’s `shortest_path_length()`** to compute the distance (in meters) between two nodes along the road network.
   * We decorate it with **`@lru_cache`** to store previously computed results — so repeated origin–destination pairs are retrieved instantly instead of recalculated.
   * This caching drastically speeds up distance computations, especially when analyzing millions of trips.

2. **We then define `calc_route_distance()`**, which:

   * Looks up the nearest road nodes for each trip’s origin and destination cells using `cell_to_node`.
   * If both nodes exist, it calculates the **shortest driving distance** via the road network.
   * If one or both nodes are missing (e.g., cell not snapped), it falls back to computing a **great-circle distance** — a direct, “as-the-crow-flies” measurement between cell centroids.
   * Handles invalid cases safely by returning `NaN` where necessary.


In [15]:
@lru_cache(maxsize=100000)
def route_distance_cached(origin_node, dest_node):
    try:
        return ox.shortest_path_length(G_drive, origin_node, dest_node, weight='length')
    except Exception:
        return np.nan

def calc_route_distance(row):
    origin_node = cell_to_node.get(row['origin_cell'])
    dest_node = cell_to_node.get(row['dest_cell'])
    
    if origin_node is None or dest_node is None or origin_node == dest_node:
        origin = cell_coord_lookup_wgs.get(row['origin_cell'])
        dest = cell_coord_lookup_wgs.get(row['dest_cell'])
        if origin is None or dest is None:
            return np.nan
        return ox.distance.great_circle(origin[1], origin[0], dest[1], dest[0])
    
    return route_distance_cached(origin_node, dest_node)

print("Functions defined.")
print("Test calc_route_distance on first row:")
print(calc_route_distance(trips_df.iloc[1]))

Functions defined.
Test calc_route_distance on first row:
nan


This function becomes the **core engine** for later stages where we’ll compute:

* Average trip lengths across Nairobi
* Travel-time proxies
* Distance-based network analyses and fairness metrics

> By combining **network-based realism** with **fallback robustness**, we ensure every trip in our dataset can be assigned a meaningful distance measure.

#### **j. Converting to Polars and Computing Route Distances in Batches**

Here we **switch to Polars** for efficient, large-scale computation of travel distances across all trips.
Because the dataset may contain **hundreds of thousands of trips**, we handle the calculations in **manageable batches** to avoid memory bottlenecks.

1. **We start by converting** our Pandas `trips_df` to a **Polars DataFrame (`trips_pl`)**.

   * Polars is faster and more memory-efficient for heavy iterative workloads.
   * It supports parallel operations and seamless integration with Python functions.

2. **We define batching parameters:**

   * `chunk_size = 10,000` ensures each batch fits comfortably in memory.
   * The total number of chunks `n_chunks` is calculated to cover the entire dataset.

3. **We process trips in chunks:**

   * For each batch, we convert it temporarily back to Pandas (`chunk_pd`) so we can apply the Python-based `calc_route_distance()` function.
   * Each trip gets a computed distance (`distance_m_temp`) using the cached network route or great-circle fallback.
   * We collect the results into a list of Polars Series for efficient concatenation afterward.

4. **We merge all distance results** into the main Polars DataFrame under the column `distance_m`.

   * We then filter out rows with missing (`NaN`) distances; those likely correspond to unmapped or invalid cells.


In [16]:
# STEP 6: Convert to Polars and Batch Compute Distances =====
print("Converting to Polars for routing...")
trips_pl = pl.from_pandas(trips_df.reset_index(drop=True))

chunk_size = 10000
n_chunks = (len(trips_pl) + chunk_size - 1) // chunk_size
distance_series = []

for i in tqdm(range(n_chunks), desc="Routing chunks"):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, len(trips_pl))
    chunk_pl = trips_pl.slice(start, end - start)
    
    chunk_pd = chunk_pl.to_pandas()
    chunk_pd['distance_m_temp'] = chunk_pd.apply(calc_route_distance, axis=1)
    
    distance_series.append(pl.Series(chunk_pd['distance_m_temp']))

trips_pl = trips_pl.with_columns(pl.concat(distance_series).alias('distance_m'))
trips_pl = trips_pl.filter(pl.col('distance_m').is_not_null())
print("Distances computed and filtered.")
print(f"Trips after NaN drop: {len(trips_pl):,}")

Converting to Polars for routing...


Routing chunks: 100%|██████████| 116/116 [00:22<00:00,  5.20it/s]

Distances computed and filtered.
Trips after NaN drop: 14,202


> This batching approach ensures we can compute **millions of road-based distances** reliably — maintaining speed, efficiency, and full reproducibility across runs.

By the end of this step, every valid trip in the dataset now includes a **realistic, route-based travel distance** (in meters), setting us up for spatial and temporal flow analysis in the next section.


#### **k. Benchmark Calibration – Congestion-Aware Speed and Duration Modeling**

Here we **move beyond static travel assumptions** and introduce a **realistic, congestion-aware traffic model** calibrated for **Nairobi’s unique mobility conditions**.
The goal is to simulate **variable travel speeds and durations** that better reflect **real-world congestion dynamics**, enabling more accurate predictive modeling (e.g., for LSTM training or travel time forecasting).

**1. Defining Time-of-Day Congestion Profiles**

We begin by setting up **empirical speed profiles** for different times of day — based on aggregated data from **TomTom Traffic Index** and **Numbeo Nairobi congestion reports**.

* **Night (00:00–06:00)** → Free flow, 35 km/h average
* **Morning Rush (06:00–09:00)** → Heavy congestion, 15 km/h average
* **Midday (10:00–14:00)** → Moderate flow, 25 km/h average
* **Afternoon Rush (15:00–19:00)** → Heaviest congestion, ~12 km/h
* **Evening (20:00–23:00)** → Clearing traffic, ~28 km/h

Here, we map each hour to a **profile** that defines both the **base speed** and a **variance window** (to simulate unpredictability).

**2. Distance-Based Speed Adjustment**

We then adjust the base speed using a **distance factor** — since trip length influences exposure to congestion and intersections:

* **Short trips (<1 km)** → Slower (more intersections and signal stops)
* **Medium trips (1–3 km)** → Neutral baseline
* **Long trips (>3 km)** → Slightly faster (more highway segments)

This creates a **realistic scaling** between micro-movements and cross-city travel.

**3. Applying the Realistic Speed Model**

We next calculate each trip’s **expected speed (`speed_kmh`)** by combining:

* The **time-of-day base speed**
* A **randomized variation factor (±15%)**
* The **distance-based adjustment**

We cap final speeds within **urban realistic bounds (5–60 km/h)** to avoid outliers.

**4. Deriving Duration and Congestion Level**

Using the final speed, we compute **trip duration (`duration_h`)** and classify trips into **Google Maps–style congestion levels**:

* `free_flow` (≥30 km/h)
* `moderate` (20–29 km/h)
* `congested` (10–19 km/h)
* `heavily_congested` (<10 km/h)

This gives both **quantitative** (speed/duration) and **qualitative** (congestion category) insights for each trip.

**5. Normalized Speed Metric**

Finally, we calculate a **normalized speed ratio**:
$$  
\text{speed\_ratio} = \frac{\text{speed\_kmh}}{35}
  $$
where 35 km/h represents **free-flow benchmark speed**.
This normalized value (0–1 scale) is useful as a **feature input for time-series and deep learning models**, ensuring stability and comparability.


In [17]:
# 1. Define Time-of-Day Congestion Profiles
# Based on Nairobi traffic patterns (TomTom/Numbeo data)
congestion_profiles = {
    'night': {'hours': list(range(0, 6)), 'base_speed': 35, 'variance': 5},      # 00:00-06:00: Free flow
    'morning_rush': {'hours': [6, 7, 8, 9], 'base_speed': 15, 'variance': 8},    # 06:00-09:00: Heavy congestion
    'midday': {'hours': [10, 11, 12, 13, 14], 'base_speed': 25, 'variance': 6},  # 10:00-14:00: Moderate
    'afternoon_rush': {'hours': [15, 16, 17, 18, 19], 'base_speed': 12, 'variance': 7}, # 15:00-19:00: Heaviest
    'evening': {'hours': [20, 21, 22, 23], 'base_speed': 28, 'variance': 5}      # 20:00-23:00: Clearing
}

# Create hour-to-profile mapping
hour_to_profile = {}
for profile_name, config in congestion_profiles.items():
    for h in config['hours']:
        hour_to_profile[h] = profile_name

# 2. Distance-Based Speed Adjustment
# Short trips (<1km) are more affected by traffic signals/congestion
# Long trips (>3km) average out with some highway segments
def distance_speed_factor(dist_m):
    """Adjust speed based on trip distance"""
    dist_km = dist_m / 1000
    if dist_km < 1.0:
        return 0.75  # Short trips: 25% slower (more intersections)
    elif dist_km > 3.0:
        return 1.15  # Long trips: 15% faster (highway segments)
    else:
        return 1.0   # Medium trips: base speed

# 3. Apply Realistic Speed Model
def calculate_realistic_speeds(trips_pl):
    """
    Calculate speeds with congestion variation suitable for LSTM prediction
    
    Returns:
    - speed_kmh: Actual estimated speed
    - congestion_level: Categorical label for classification
    - speed_ratio: Speed/free_flow_speed (0-1 scale for normalization)
    """
    
    # Map hours to congestion profiles
    trips_pl = trips_pl.with_columns([
        pl.col('hour').map_elements(
            lambda h: hour_to_profile.get(h, 'midday'),
            return_dtype=pl.Utf8
        ).alias('time_period')
    ])
    
    # Base speed from time-of-day profile
    profile_mapping = {name: config['base_speed'] for name, config in congestion_profiles.items()}
    variance_mapping = {name: config['variance'] for name, config in congestion_profiles.items()}
    
    trips_pl = trips_pl.with_columns([
        pl.col('time_period').map_elements(
            lambda p: profile_mapping.get(p, 25),
            return_dtype=pl.Float64
        ).alias('base_speed_kmh'),
        pl.col('time_period').map_elements(
            lambda p: variance_mapping.get(p, 5),
            return_dtype=pl.Float64
        ).alias('speed_variance')
    ])
    
    # Add random variation within profile (simulates real-world variance)
    np.random.seed(42)  # For reproducibility
    n_trips = len(trips_pl)
    random_factors = np.random.normal(1.0, 0.15, n_trips)  # ±15% variation
    
    trips_pl = trips_pl.with_columns([
        pl.Series('random_factor', random_factors)
    ])
    
    # Distance-based adjustment
    trips_pl = trips_pl.with_columns([
        pl.col('distance_m').map_elements(
            distance_speed_factor,
            return_dtype=pl.Float64
        ).alias('distance_factor')
    ])
    
    # Final speed calculation
    trips_pl = trips_pl.with_columns([
        (pl.col('base_speed_kmh') * 
         pl.col('random_factor') * 
         pl.col('distance_factor')).alias('speed_kmh')
    ])
    
    # Clip to realistic bounds (5-60 km/h for urban Nairobi)
    trips_pl = trips_pl.with_columns([
        pl.col('speed_kmh').clip(5, 60).alias('speed_kmh')
    ])
    
    # Calculate duration from realistic speed
    trips_pl = trips_pl.with_columns([
        (pl.col('distance_m') / 1000 / pl.col('speed_kmh')).alias('duration_h')
    ])
    
    # 4. Add Congestion Level Categories (Google Maps style)
    trips_pl = trips_pl.with_columns([
        pl.when(pl.col('speed_kmh') >= 30)
          .then(pl.lit('free_flow'))
        .when((pl.col('speed_kmh') >= 20) & (pl.col('speed_kmh') < 30))
          .then(pl.lit('moderate'))
        .when((pl.col('speed_kmh') >= 10) & (pl.col('speed_kmh') < 20))
          .then(pl.lit('congested'))
        .otherwise(pl.lit('heavily_congested'))
        .alias('congestion_level')
    ])
    
    # 5. Speed Ratio (normalized metric for LSTM - 0 to 1 scale)
    # Speed ratio = actual_speed / free_flow_speed (35 km/h baseline)
    free_flow_speed = 35.0
    trips_pl = trips_pl.with_columns([
        (pl.col('speed_kmh') / free_flow_speed).alias('speed_ratio')
    ])
    
    return trips_pl

> **In summary:**
Here we simulate **realistic, congestion-sensitive trip behavior** — embedding Nairobi’s daily rhythm of traffic into the dataset.
This step transforms static distance data into **dynamic mobility sequences** that better capture **spatial–temporal movement patterns**, forming the bridge between **trajectory analytics** and **predictive traffic modeling**.

**We then apply** our `calculate_realistic_speeds()` function, which dynamically assigns:

   * `speed_kmh` → simulated travel speed based on time, distance, and congestion
   * `duration_h` → realistic trip duration
   * `congestion_level` → categorical traffic condition (free_flow → heavily_congested)
   * `speed_ratio` → normalized (0–1) metric for model-ready features**Finally, we convert** the resulting Polars DataFrame back to Pandas for compatibility with downstream analysis, visualization, or machine learning workflows.

In [18]:
# Apply to your trips data
trips_pl = calculate_realistic_speeds(trips_pl)

# Convert back to pandas
trips_df = trips_pl.to_pandas()

> At this point, each trip record is enriched with **context-aware mobility attributes**, turning raw trajectories into **interpretable, simulation-grade transport data**.

In [19]:
trips_df.shape

(14202, 17)

#### **l. Model 2 – Static Traffic Baseline (October 1st Snapshot)**

Here we **build a static, cell-based traffic model**, representing Nairobi’s average traffic state across a full 24-hour cycle.
This acts as a **baseline benchmark** for comparing dynamic models later (like temporal LSTM forecasts).
We focus on **cell-level aggregates**, treating each grid cell as a fixed observation point over time.

**1. Aggregating Cell-Level Metrics**

Here we **aggregate all trips** by `origin_cell` and `hour` to capture spatial-temporal variation in mobility patterns.
For each combination, we compute:

* **`avg_speed_kmh`** → Mean travel speed within the cell-hour window
* **`speed_std`** → Standard deviation to capture speed variability
* **`trip_count`** → Total trips originating in the cell during that hour
* **`congestion_level`** → Dominant congestion class (most frequent label)
* **`congestion_pct`** → % of trips under congested or heavily congested conditions
* **`ward`** → Administrative context, inherited from the grid-to-ward mapping

This provides a **quantitative signature** of how each part of Nairobi behaves hour by hour.

**2. Enriching with Spatial Coordinates**

We merge in the **cell coordinates (`lat`, `lon`)** from our earlier lookup dictionary, giving every grid cell a geospatial identity.

In [20]:
# Use ALL 24 hours from trips_df
print("\nAggregating cell-level traffic (all hours)...")

model2_traffic = trips_df.groupby(['origin_cell', 'hour']).agg({
    'speed_kmh': ['mean', 'std'],
    'agent_id': 'count',
    'congestion_level': [
        lambda x: x.mode()[0] if len(x) > 0 else 'unknown',
        lambda x: (x.isin(['congested', 'heavily_congested']).sum() / len(x) * 100)
    ],
    'origin_ward': 'first'
}).reset_index()

# Flatten column names
model2_traffic.columns = [
    'cell_id', 'hour', 'avg_speed_kmh', 'speed_std', 
    'trip_count', 'congestion_level', 'congestion_pct', 'ward'
]

# Add cell coordinates
cell_coords_df = pd.DataFrame([
    {'cell_id': cell, 'lat': coords[1], 'lon': coords[0]} 
    for cell, coords in cell_coord_lookup_wgs.items()
])

model2_traffic = model2_traffic.merge(cell_coords_df, on='cell_id', how='left')

# Reorder columns (final 8 columns)
model2_traffic = model2_traffic[[
    'cell_id',           # Primary key 1
    'hour',              # Primary key 2
    'avg_speed_kmh',     # Main metric
    'congestion_level',  # Category
    'congestion_pct',    # Percentage
    'trip_count',        # Volume
    'ward',              # Context
    'lat',               # Location
    'lon'                # Location
]]


Aggregating cell-level traffic (all hours)...


In [21]:
print(model2_traffic.shape)
model2_traffic.head(5)

(2895, 9)


,cell_id,hour,avg_speed_kmh,congestion_level,congestion_pct,trip_count,ward,lat,lon
0,77,0,21.423049,moderate,0.0,1,Kahawa West,-1.178553,36.930857
1,77,2,25.926126,moderate,0.0,1,Kahawa West,-1.178553,36.930857
2,77,3,26.857083,moderate,12.5,8,Kahawa West,-1.178553,36.930857
3,77,4,29.185721,free_flow,0.0,2,Kahawa West,-1.178553,36.930857
4,77,5,25.848644,congested,50.0,2,Kahawa West,-1.178553,36.930857


In [22]:
model2_traffic.to_parquet('/home/dataopske/Desktop/jav/data/processed/model2_traffic.parquet', index=False)

This table shows the **cell-level traffic summary** from our static baseline (Model 2).
Each row represents one **grid cell–hour pair**, summarizing average traffic conditions in that location and time window.

| Column               | Description                                                       |
| :------------------- | :---------------------------------------------------------------- |
| **cell_id**          | Unique grid cell identifier (spatial unit of analysis)            |
| **hour**             | Hour of day (0–23)                                                |
| **avg_speed_kmh**    | Mean travel speed within the cell at that hour                    |
| **congestion_level** | Dominant traffic condition (e.g., free_flow, moderate, congested) |
| **congestion_pct**   | % of trips under congested or heavily congested states            |
| **trip_count**       | Number of observed trips originating in that cell-hour            |
| **ward**             | Administrative ward containing the cell                           |
| **lat**, **lon**     | Geographic coordinates of the cell centroid                       |

From the preview above (cell 77 – *Kahawa West*), we can see:

* Speeds vary by hour, ranging from ~21 km/h at midnight to ~29 km/h near dawn.
* Congestion level shifts between **moderate**, **free flow**, and **congested**, reflecting natural traffic fluctuations overnight.
* The cell maintains a consistent spatial reference (`lat=-1.178553`, `lon=36.930857`) across all time intervals.

This confirms that **Model 2** captures hourly traffic dynamics at fine spatial resolution, ready for mapping, analysis, or predictive modeling.


In [23]:
# Add this section to your initial notebook (after saving model2_traffic.parquet)
# It aggregates hourly data to cell-level daily/peak metrics for faster lookup and fewer NaNs

print("Aggregating Model 2 to cell-level daily metrics...")

# Load if needed, but assuming model2_traffic is already in memory as pd.DataFrame
# If not: model2_traffic = pd.read_parquet('/home/dataopske/Desktop/jav/data/processed/model2_traffic.parquet')

# Define peak hours (match your constants)
PEAK_HOURS = [6, 7, 8, 9, 17, 18, 19, 20]

# SPEEDUP: Pre-compute peak/offpeak separately for efficiency (avoids slow lambdas in agg)
peak_data = model2_traffic[model2_traffic['hour'].isin(PEAK_HOURS)]
offpeak_data = model2_traffic[~model2_traffic['hour'].isin(PEAK_HOURS)]

avg_speed_peak = peak_data.groupby('cell_id')['avg_speed_kmh'].mean().reset_index(name='avg_speed_peak')
avg_speed_offpeak = offpeak_data.groupby('cell_id')['avg_speed_kmh'].mean().reset_index(name='avg_speed_offpeak')
congestion_pct_peak = peak_data.groupby('cell_id')['congestion_pct'].mean().reset_index(name='congestion_pct_peak')
trip_count_peak = peak_data.groupby('cell_id')['trip_count'].sum().reset_index(name='trip_count_peak')

# Now base daily agg (simpler, no lambdas)
model2_daily = model2_traffic.groupby('cell_id').agg({
    'avg_speed_kmh': 'mean',
    'congestion_pct': 'mean',
    'trip_count': 'sum',
    'ward': 'first',
    'lat': 'first',
    'lon': 'first'
}).reset_index()

# Rename base columns
model2_daily = model2_daily.rename(columns={
    'avg_speed_kmh': 'avg_speed_daily',
    'congestion_pct': 'congestion_pct_daily',
    'trip_count': 'trip_count_daily'
})

# Merge the pre-computed peak/offpeak
model2_daily = model2_daily.merge(avg_speed_peak, on='cell_id', how='left')
model2_daily = model2_daily.merge(avg_speed_offpeak, on='cell_id', how='left')
model2_daily = model2_daily.merge(congestion_pct_peak, on='cell_id', how='left')
model2_daily = model2_daily.merge(trip_count_peak, on='cell_id', how='left')

# For demand_variability_cv: std/mean of hourly trip_count per cell
hourly_trips = model2_traffic.groupby(['cell_id', 'hour'])['trip_count'].sum().reset_index()
hourly_trips_pivot = hourly_trips.pivot(index='cell_id', columns='hour', values='trip_count').fillna(0)
cv_per_cell = hourly_trips_pivot.std(axis=1) / (hourly_trips_pivot.mean(axis=1) + 1)  # +1 to avoid div0
model2_daily['demand_variability_cv'] = cv_per_cell.reindex(model2_daily['cell_id']).values

# For dominant_congestion_level: mode of congestion_level per cell
mode_cong = model2_traffic.groupby('cell_id')['congestion_level'].agg(lambda x: x.mode()[0] if len(x) > 0 else 'unknown').reset_index()
mode_cong = mode_cong.rename(columns={'congestion_level': 'dominant_congestion_level'})
model2_daily = model2_daily.merge(mode_cong, on='cell_id', how='left')

# Reorder columns
model2_daily = model2_daily[[
    'cell_id', 'lat', 'lon', 'ward',
    'avg_speed_daily', 'avg_speed_peak', 'avg_speed_offpeak',
    'congestion_pct_daily', 'congestion_pct_peak',
    'trip_count_daily', 'trip_count_peak',
    'demand_variability_cv', 'dominant_congestion_level'
]]

print(f"Aggregated to {len(model2_daily)} cells")
model2_daily.head()

# Save the daily aggregated version
model2_daily.to_parquet('/home/dataopske/Desktop/jav/data/processed/model2_traffic_daily.parquet', index=False)
print("✓ Saved model2_traffic_daily.parquet")

Aggregating Model 2 to cell-level daily metrics...
Aggregated to 170 cells
✓ Saved model2_traffic_daily.parquet


#### **m. Loading GTFS Transit Data (Digital Matatus 2019)**

We **load public transport GTFS data** from the **Digital Matatus (2019)** feed to integrate real-world bus and matatu routes with our Nairobi mobility model.

1. **We import** the GTFS ZIP feed using `gtfs_kit`, which automatically parses route, trip, stop, and shape tables.
2. **We extract** key components — routes, trips, stop times, stops, and shapes — for later spatial joins with our road and grid data.
3. **We print a quick summary** to confirm successful load and preview route identifiers.


In [24]:
# After: graph_path = "nairobi_drive.graphml" block
# Add this:

print("Loading GTFS data...")
# import GTFS feed data
feed_path = '/home/dataopske/Desktop/jav/data/raw/digitalmatatu/GTFS_FEED_2019.zip'
feed = gk.read_feed(feed_path,dist_units='km')
gtfs_routes = feed.routes
gtfs_trips = feed.trips
gtfs_stop_times = feed.stop_times
gtfs_stops = feed.stops
gtfs_shapes = feed.shapes

print(f"✓ GTFS loaded: {len(gtfs_routes)} routes, {len(gtfs_stops)} stops")
print(f"  Route IDs: {gtfs_routes['route_id'].unique()[:5]}...")

Loading GTFS data...
✓ GTFS loaded: 136 routes, 4284 stops
  Route IDs: <StringArray>
['10000107D11', '10000114011', '10000116011', '10100011A11', '10200010811']
Length: 5, dtype: string...


> This dataset will help **overlay real transit corridors** on top of the simulated traffic network, allowing us to compare **mode coverage**, **equity**, and **service accessibility**.

#### **n. Route ETA Calculator (Using Model 2 Baseline)**

We define a helper function to **estimate travel time (ETA)** for any GTFS bus or matatu route at a given hour, using the **cell-based speed data** from **Model 2**.

1. **We start by selecting** a GTFS route and its corresponding trip and stops.
2. **We compute stop-to-stop segments**, finding each segment’s **midpoint** and matching it to the **nearest traffic cell** from Model 2.
3. **We retrieve** the average traffic speed for that cell and hour — or use a fallback speed (15 km/h) if no data is available.
4. **We calculate segment distance** using the **Haversine formula**, then convert it into **segment travel time (minutes)**.
5. **We sum all segment ETAs** to estimate the **total route travel time** under typical conditions for that hour.


In [25]:
# ROUTE ETA CALCULATOR (Uses Model 2 data)
def calculate_route_eta(route_id, hour):
    """
    Calculate ETA for a GTFS route at given hour
    Uses cell-level speeds from Model 2
    """
    # Get route's trip
    route_trips = gtfs_trips[gtfs_trips['route_id'] == route_id]
    if len(route_trips) == 0:
        return None
    
    trip_id = route_trips.iloc[0]['trip_id']
    
    # Get stops in sequence
    stops = gtfs_stop_times[gtfs_stop_times['trip_id'] == trip_id].sort_values('stop_sequence')
    stop_ids = stops['stop_id'].tolist()
    
    # Get stop coordinates
    stop_coords = gtfs_stops[gtfs_stops['stop_id'].isin(stop_ids)][['stop_id', 'stop_lat', 'stop_lon']]
    
    total_eta = 0
    
    # Calculate segment-by-segment
    for i in range(len(stop_ids) - 1):
        from_stop = stop_coords[stop_coords['stop_id'] == stop_ids[i]].iloc[0]
        to_stop = stop_coords[stop_coords['stop_id'] == stop_ids[i+1]].iloc[0]
        
        # Find nearest cell to midpoint
        mid_lat = (from_stop['stop_lat'] + to_stop['stop_lat']) / 2
        mid_lon = (from_stop['stop_lon'] + to_stop['stop_lon']) / 2
        
        # Find closest cell
        distances = np.sqrt(
            (model2_traffic['lat'] - mid_lat)**2 + 
            (model2_traffic['lon'] - mid_lon)**2
        )
        nearest_idx = distances.idxmin()
        
        # Get speed at this hour
        speed = model2_traffic[
            (model2_traffic['cell_id'] == model2_traffic.loc[nearest_idx, 'cell_id']) &
            (model2_traffic['hour'] == hour)
        ]['avg_speed_kmh'].values
        
        if len(speed) == 0:
            speed = 15  # Default fallback
        else:
            speed = speed[0]
        
        # Calculate distance (haversine)
        from math import radians, cos, sin, asin, sqrt
        lon1, lat1, lon2, lat2 = map(radians, [from_stop['stop_lon'], from_stop['stop_lat'], 
                                                 to_stop['stop_lon'], to_stop['stop_lat']])
        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        distance_km = 2 * asin(sqrt(a)) * 6371
        
        # Segment ETA
        segment_eta = (distance_km / speed) * 60  # minutes
        total_eta += segment_eta
    
    return total_eta

> This function lets us **simulate route performance dynamically**, comparing public transport ETAs across different times of day — a key step for identifying congestion hotspots and service reliability issues.


#### **t. Example – Estimating Route ETA**

We **test the ETA calculator** on a real GTFS route (`107D – Ruaka–Ruiru`) using the **Model 2 traffic speeds**.
We pass an hour parameter (`hour=15`) to simulate afternoon traffic conditions and estimate how long the route would take end-to-end.


In [26]:
# Example usage
route_107d_eta = calculate_route_eta('70101013301', hour=15)
print(f"\nRoute 107D (Ruaka-Ruiru) ETA at 9am: {route_107d_eta:.0f} minutes")
print("(Based on Oct 1 cell-level speeds)")


Route 107D (Ruaka-Ruiru) ETA at 9am: 30 minutes
(Based on Oct 1 cell-level speeds)


> This gives us a quick, interpretable result; an **estimated total travel time (in minutes)** for that route under typical congestion around **3 PM**, based on the **October 1 static baseline**.